# Analysis of the Conservation State of Ancient Books

**Information Systems and Business Intelligence** — A.Y. 2025/2026

Reproducible analysis (Phase 2). Run the setup cell, then **Runtime → Run all**.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def _run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# Colab / Jupyter: clone repo if src/ is missing
if not Path('src').exists():
    _run('git clone https://github.com/kimias21/book-conservation-bi.git')
    os.chdir('book-conservation-bi')

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_run('pip install -q -r requirements.txt')
if not Path('data/processed/integrated_heritage_books.csv').exists():
    _run(f'{sys.executable} -m src.data_integration')

import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import INTEGRATED_CSV, DATA_PROCESSED
from src.conservation_index import add_conservation_columns

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('Data file exists:', INTEGRATED_CSV.exists())


## 1. Loading and pre-processing

In [ ]:
df = pd.read_csv(INTEGRATED_CSV)
print(df.shape)
df.head()

In [ ]:
TAXONOMY = {'parchment':'Parchment','laid_paper':'Laid paper','industrial_paper':'Industrial paper','mixed':'Mixed support'}
df['support_material_label'] = df['support_material'].map(TAXONOMY)
for c in ['flood_event','hvac_failure_event','restoration_recent']:
    df[c] = df[c].astype(bool)
print('Missing:', df.isnull().sum().sum())
df.describe().T

Exploratory analysis of CRI by material, century, and environmental variables.


## 2. Exploratory analysis

In [ ]:
# CRI distribution
df['conservation_risk_index'].hist(bins=30)
plt.title('CRI distribution')
plt.xlabel('Conservation Risk Index (0–100)')
plt.tight_layout()
plt.show()

# CRI by support material
sns.boxplot(data=df, x='support_material_label', y='conservation_risk_index')
plt.title('CRI by support material')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Correlation matrix
corr_cols = ['conservation_risk_index', 'avg_relative_humidity_pct',
             'avg_temperature_c', 'age_stress', 'material_stress']
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation matrix (selected variables)')
plt.tight_layout()
plt.show()

## 3. Conservation Risk Index

In [ ]:
df = add_conservation_columns(df)

# Risk level distribution
print(df['risk_level'].value_counts())

# Correlation between CRI and expert-assessed condition score
# (Pearson r validates that CRI tracks real deterioration)
df[['conservation_risk_index', 'observed_condition_score']].corr()

## 4. Models

In [ ]:
from src.conservation_index import CRI_ELEVATED_THRESHOLD

groups = [g['conservation_risk_index'].values for _, g in df.groupby('support_material')]
print('ANOVA F,p:', stats.f_oneway(*groups))

# Target: elevated risk — uses shared threshold from conservation_index.py
# (CRI_ELEVATED_THRESHOLD = 45; consistent with risk_label categories)
df['elevated_risk'] = (df['conservation_risk_index'] >= CRI_ELEVATED_THRESHOLD).astype(int)
print(f'Elevated risk count (CRI >= {CRI_ELEVATED_THRESHOLD}): {df["elevated_risk"].sum()}')

feature_cols = ['century', 'avg_temperature_c', 'avg_relative_humidity_pct', 'avg_light_lux',
                'air_quality_index', 'support_material', 'ink_type', 'binding_type', 'site_type']
X_feat, y = df[feature_cols], df['elevated_risk']
cat = ['support_material', 'ink_type', 'binding_type', 'site_type']
num = [c for c in feature_cols if c not in cat]
prep = ColumnTransformer([('num', StandardScaler(), num),
                          ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
split_args = dict(test_size=0.25, random_state=42)
if y.sum() >= 10 and (len(y) - y.sum()) >= 10:
    split_args['stratify'] = y
X_train, X_test, y_train, y_test = train_test_split(X_feat, y, **split_args)
rf = Pipeline([('prep', prep),
               ('clf', RandomForestClassifier(200, random_state=42, class_weight='balanced'))])
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), zero_division=0))

### Geographic and temporal visualisations

In [ ]:
ts_path = DATA_PROCESSED / 'environmental_timeseries.csv'
if not ts_path.exists():
    print('WARNING: environmental_timeseries.csv not found — re-run src.data_integration')
else:
    ts = pd.read_csv(ts_path)
    fig, ax = plt.subplots(figsize=(10, 4))
    for sid, g in ts.groupby('site_id'):
        ax.plot(g['month'], g['avg_relative_humidity_pct'], label=sid)
    ax.set_ylabel('RH %')
    ax.set_title('Temporal evolution of relative humidity by site')
    ax.legend(ncol=4, fontsize=7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### Clustering risk profiles

In [ ]:
# Cluster volumes by stress profile (K=4 chosen by elbow; see report §4.2)
X_clust = df[['environmental_stress', 'material_stress', 'age_stress', 'event_stress']]
km = KMeans(4, random_state=42, n_init=10).fit(StandardScaler().fit_transform(X_clust))
df['cluster'] = km.labels_
print('Mean CRI per cluster:')
print(df.groupby('cluster')['conservation_risk_index'].mean().sort_values())

## 5. Discussion
See report/final_report.md for decision-maker narrative, assumptions, and limitations.